In [1]:
import pandas as pd
import numpy as np
import heapq

In [2]:
df = pd.read_excel('dados/base_analise.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro1/Dados-processados/2_df_recodificado.parquet')
sinan = sinan[sinan['ANO'].isin(range(2019,2025))]

## 1. Caracterização epidemiológica do escorpionismo
### Parte 1
1.1 Total de acidentes escorpiônicos no período de 2019–2024.  
1.2. Distribuição anual dos casos.  
1.3 Incidência média anual por 100.000 habitantes.  
1.4 Distribuição dos casos segundo faixas etárias de interesse.  
1.5 Número e proporção de casos MG segundo a definição operacional.
### Parte 2  
1.6 Distribuição dos principais indicadores: por município; por região.  
1.7 Identificação da heterogeneidade territorial da carga do escorpionismo.  
1.8 Produto principal: Tabela de caracterização epidemiológica, complementada por mapa da distribuição territorial do agravo.


In [3]:
# Parte 1
print(
      '---------------------------------------------------------------------------- \n',
      '1.1 Total de acidentes entre 2019 e 2024:', df['TOTAL_CASOS'].sum(), '\n\n',

      
      '---------------------------------------------------------------------------- \n'
      '1.2 Distribuição anual dos casos:', sinan['ANO'].value_counts(), '\n\n',


      '---------------------------------------------------------------------------- \n'
      '1.3 Incidência média anual por 100 mil hab.:', df['INC_MEDIA_ANUAL'].describe(), '\n\n',


      '---------------------------------------------------------------------------- \n'
      '1.4 Distribuição dos casos por faixa etária:', '\n\n',
      'Total até 10 anos:', df['TOTAL_10A'].sum(), f"({df['TOTAL_10A'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)", '\n',

      'Total de 11 a 59  anos:', df['TOTAL_11A59'].sum(), f"({df['TOTAL_11A59'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)" '\n',

      'Total 60+:', df['TOTAL_60'].sum(),  f"({df['TOTAL_60'].sum() / df['TOTAL_CASOS'].sum()*100:.1f}%)" '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.5 Numero e proporção de casos MG:', df['MG'].sum(), f"({df['MG'].sum() / df['TOTAL_CASOS'].sum() * 100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.6 Numero e proporção de MG ate 10 anos entre o total MG:', df['MG_10'].sum(), f"({df['MG_10'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.7 Numero e proporção de MG ate 11A59 anos entre o total MG:', df['MG_11A59'].sum(), f"({df['MG_11A59'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',
      
      '---------------------------------------------------------------------------- \n',
      '1.8 Numero e proporção de MG 60+ entre o total MG:', df['MG_60'].sum(), f"({df['MG_60'].sum()/df['MG'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.9 Numero de óbitos e prop. em relação ao total de casos:', df['OBITOS'].sum(), f"({df['OBITOS'].sum()/df['TOTAL_CASOS'].sum()*100:.3f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.10 Num. e prop obitos 10 anos em relacao ao total obitos:', df['OBITOS_10'].sum(), f"({df['OBITOS_10'].sum()/df['OBITOS'].sum()*100:.1f}%)", '\n\n',


      '---------------------------------------------------------------------------- \n',
      '1.11 Num. e prop obitos 11 a 59 anos em relacao ao total obitos:', df['OBITOS_11A59'].sum(), f"({df['OBITOS_11A59'].sum()/df['OBITOS'].sum()*100:.1f}%)", '\n\n',
      
      
      '---------------------------------------------------------------------------- \n',
      '1.11 Num. e prop obitos 60+ em relacao ao total obitos:', df['OBITOS_60'].sum(), f"({df['OBITOS_60'].sum()/df['OBITOS'].sum()*100:.1f}%)"

      )



---------------------------------------------------------------------------- 
 1.1 Total de acidentes entre 2019 e 2024: 239086 

 ---------------------------------------------------------------------------- 
1.2 Distribuição anual dos casos: ANO
2023    50125
2022    43619
2024    42148
2020    35275
2021    34437
2019    33484
Name: count, dtype: int64 

 ---------------------------------------------------------------------------- 
1.3 Incidência média anual por 100 mil hab.: count     645.000000
mean      295.713442
std       306.888537
min         0.000000
25%        50.795801
50%       192.607163
75%       445.729430
max      1590.079874
Name: INC_MEDIA_ANUAL, dtype: float64 

 ---------------------------------------------------------------------------- 
1.4 Distribuição dos casos por faixa etária: 

 Total até 10 anos: 22610 (9.5%) 
 Total de 11 a 59  anos: 166395 (69.6%)
 Total 60+: 50081 (20.9%)

 ---------------------------------------------------------------------------- 
 1.

In [4]:
# 'População, acidentes, incidência média anual, casos ate 10 anos(n/%), mg(n/%)'

import pandas as pd

# ============================================================
# TABELA EPIDEMIOLÓGICA POR REGIÃO
# ============================================================

# 1. Agregação regional
tabela_regiao = (
    df
    .groupby("REGIAO", as_index=False)
    .agg(
        POPULACAO=("POP_GERAL", "sum"),
        CASOS=("TOTAL_CASOS", "sum"),
        ATE_10=("TOTAL_10A", "sum"),
        MG=("MG", "sum"),
        OBITOS=("OBITOS", "sum")
    )
)

# 2. Linha total do Estado
total_estado = pd.DataFrame({
    "REGIAO": ["Estado de São Paulo"],
    "POPULACAO": [df["POP_GERAL"].sum()],
    "CASOS": [df["TOTAL_CASOS"].sum()],
    "ATE_10": [df["TOTAL_10A"].sum()],
    "MG": [df["MG"].sum()],
    "OBITOS": [df["OBITOS"].sum()]
})

# 3. Acrescentar o total às regiões e calcular os indicadores
tabela_regiao = (
    pd.concat(
        [tabela_regiao, total_estado],
        ignore_index=True
    )
    .assign(
        INC_MEDIA_ANUAL=lambda x:
            x["CASOS"] / (x["POPULACAO"] * 6) * 100000,

        ATE_10_N_PCT=lambda x:
            x["ATE_10"].astype(str) + " (" +
            (x["ATE_10"] / x["CASOS"] * 100)
            .round(1).astype(str) + "%)",

        MG_N_PCT=lambda x:
            x["MG"].astype(str) + " (" +
            (x["MG"] / x["CASOS"] * 100)
            .round(1).astype(str) + "%)",

        OBITOS_N_LET=lambda x:
            x["OBITOS"].astype(str) + " (" +
            (x["OBITOS"] / x["CASOS"] * 100)
            .round(3).astype(str) + "%)"
    )
    [[
        "REGIAO",
        "POPULACAO",
        "CASOS",
        "INC_MEDIA_ANUAL",
        "ATE_10_N_PCT",
        "MG_N_PCT",
        "OBITOS_N_LET"
    ]]
    .round({
        "INC_MEDIA_ANUAL": 1
    })
)

tabela_regiao


tabela_regiao.to_excel('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro2/resultados/tabela_regiao_epidemio.xlsx')


In [32]:
df.columns

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'REGIAO', 'PESA', 'MUNI',
       'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA', 'LON_PESA',
       'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60', 'POP_GERAL',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60', 'LEVE', 'MG',
       'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59', 'MG_11A59', 'LEVE_60',
       'MG_60', 'TOTAL_AMPOLAS', 'OBITOS', 'OBITOS_10', 'OBITOS_11A59',
       'OBITOS_60', 'N_PESA', 'CAT_TEMPO', 'PROP_POP_0A10', 'PROP_POP_60MAIS',
       'LOG_POP', 'PROP_MG', 'INCID_MEDIA_GERAL', 'INC_MEDIA_ANUAL',
       'INCID_0A10', 'INCID_60MAIS', 'PROP_CASOS_0A10', 'PROP_CASOS_60MAIS',
       'TAXA_MG_100MIL', 'CLASS_IG', 'SORO_IG', 'EVOL_IG', 'PROP_CLASS_IGN',
       'PROP_SORO_IGN', 'PROP_EVOL_IGN', 'BAIXO_N', 'TEMPO_50', 'CARGA_MG'],
      dtype='str')

In [18]:
# Parte 2
print(
    '---------------------------------------------------------------------------- \n',
          '1.6 Principais indicadores: por município; por região', '\n\n',
          '---------------------------------------------------------------------------- \n',
          '1.7 Identificação da heterogeneidade territorial da carga do escorpionismo', '\n\n',
          '---------------------------------------------------------------------------- \n',
          '1.8 Produto principal: Tabela de caracterização epidemiológica, complementada por mapa da distribuição territorial do agravo.''\n\n',
)

---------------------------------------------------------------------------- 
 1.6 Principais indicadores: por município; por região 

 ---------------------------------------------------------------------------- 
 1.7 Identificação da heterogeneidade territorial da carga do escorpionismo 

 ---------------------------------------------------------------------------- 
 1.8 Produto principal: Tabela de caracterização epidemiológica, complementada por mapa da distribuição territorial do agravo.




In [19]:
df[['MUNI','INC_MEDIA_ANUAL','INCID_MEDIA_GERAL']]

,MUNI,INC_MEDIA_ANUAL,INCID_MEDIA_GERAL
0,ALTO ALEGRE,1247.568299,1497.08
1,AVANHANDAVA,355.072464,426.09
2,BARBOSA,1306.625223,1567.95
3,BENTO DE ABREU,520.637310,624.76
4,BILAC,580.875782,697.05
...,...,...,...
640,IPIGUA,621.164496,745.40
641,ONDA VERDE,898.078530,1077.69
642,PIRACICABA,311.442285,373.73
643,RIO DAS PEDRAS,154.378549,185.25


## 2. Caracterização da rede PESA
* Número de PESA identificados.
* Número e proporção de municípios:
    * com PESA local;
    * sem PESA local.
* Distribuição dos PESA segundo regiões.
* População residente em municípios com e sem PESA local.
* Número/proporção de acidentes ocorridos em municípios com e sem PESA local.
* Número/proporção de MG em municípios com e sem PESA local.
* Produto principal: Mapa da distribuição territorial da rede PESA.


# 3. Acessibilidade geográfica aos PESA
* População de análise: municípios sem PESA local.
* Número de municípios dependentes de deslocamento intermunicipal.
* Tempo estimado até o PESA de referência:
    * mediana;
    * intervalo interquartil;
    * mínimo e máximo.
* Distribuição dos municípios segundo estratos de tempo.
* Distribuição regional dos tempos de deslocamento.
* População residente em cada estrato de acessibilidade.
* Número de acidentes e MG em cada estrato de tempo.
* Identificação dos municípios/regiões com maiores tempos estimados.
* Produto principal: Mapa do tempo estimado de deslocamento até o PESA de referência.


## 4. Sobreposição entre carga epidemiológica e acessibilidade
* População de análise: municípios sem PESA local.
* Construção de duas variáveis categóricas:
    * carga acumulada de MG entre 2019–2024: menor, intermediária e maior;
    * tempo estimado até o PESA: menor, intermediário e maior.
* Construção da matriz bivariada 3 × 3, resultando em nove perfis territoriais.
* Para cada perfil:
    * número de municípios;
    * população;
    * total de acidentes;
    * número de MG;
    * participação na carga de MG.
* Identificação dos municípios classificados simultaneamente como maior carga de MG + maior tempo de deslocamento.
* Avaliação estatística da sobreposição:
* qui-quadrado de Pearson;
* V de Cramér;
* frequências observadas e esperadas;
* resíduos padronizados.
* Interpretação como sobreposição territorial entre carga e acessibilidade, e não como efeito do tempo sobre a gravidade.
* Produto principal: Mapa bivariado carga de MG × tempo de acesso, acompanhado da matriz 3 × 3.
